In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Drive 경로는 실제 업로드 위치로 바꾸세요
DRIVE_DATA = '/content/drive/MyDrive/Project/Human Fall Detection -relabel-.v11.yolov11.zip'

Mounted at /content/drive


In [2]:
!unzip -q "/content/drive/MyDrive/Project/Human Fall Detection -relabel-.v1i.yolov11.zip" -d /content/raw_data


In [3]:
import yaml
with open('/content/raw_data/data.yaml') as f:  # 안 보이면 하위 폴더 이름을 경로에 추가
    print(yaml.safe_load(f)['names'])

['fall', 'non-fall']


In [4]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 8.7 MB/s eta 0:00:00


In [5]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')   # COCO 사전학습 가중치 자동 다운로드
model.train(
    data='/content/raw_data/data.yaml',
    imgsz=320,        # 파이 배포 해상도와 반드시 동일하게
    epochs=100,
    patience=20,      # 20에폭 동안 개선 없으면 조기 종료
    batch=32,
    degrees=0.0,      # 회전 증강 금지 — 서 있는 사람이 회전하면 누운 사람처럼 보여 라벨 의미가 깨짐
    flipud=0.0,       # 상하반전도 같은 이유로 금지
    fliplr=0.5,        # 좌우반전은 안전
    project='/content/runs', name='yolo11n_fall'
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.157 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/raw_data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, f

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79d5e09609f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [6]:
import shutil
shutil.copytree('/content/runs', '/content/drive/MyDrive/runs_backup', dirs_exist_ok=True)

'/content/drive/MyDrive/runs_backup'

In [7]:
best = model.trainer.save_dir / 'weights' / 'best.pt'
m = YOLO(str(best))
r = m.val(data='/content/raw_data/data.yaml', split='test', imgsz=320)
print(f"mAP50: {r.box.map50:.3f}")

Ultralytics 8.4.157 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1701.3±516.8 MB/s, size: 46.6 KB)
val: Scanning /content/raw_data/test/labels... 2152 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2152/2152 2.3Kit/s 0.9s
val: New cache created: /content/raw_data/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 135/135 11.2it/s 12.1s
                   all       2152       2152      0.907       0.94      0.929      0.619
                  fall        793        793      0.882      0.945      0.893      0.602
              non-fall       1359       1359      0.932      0.935      0.965      0.635
Speed: 0.2ms preprocess, 1.3ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /content/runs/detect/val
mAP50: 0.929


In [8]:
m.export(format='tflite', imgsz=320, int8=True, data='/content/raw_data/data.yaml')

WARNING ⚠️ 'int8' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ format='tflite' is deprecated as of 8.4.83 and has been replaced by the unified Google LiteRT format. Exporting format='litert' instead. See https://docs.ultralytics.com/integrations/litert
Ultralytics 8.4.157 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n summary (fused): 100 layers, 2,582,542 parameters, 0 gradients, 1.6 GFLOPs

PyTorch: starting from '/content/runs/yolo11n_fall/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 6, 2100) (5.2 MB)
LiteRT: collecting INT8 calibration images from 'data=/content/raw_data/data.yaml'
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1488.8±402.4 MB/s, size: 45.6 KB)
val: Scanning /content/raw_data/valid/labels.cache... 859 images, 0 backgrounds,

/usr/local/lib/python3.13/dist-packages/torchao/quantization/quant_api.py:1558: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.



LiteRT: starting export with litert_torch 0.9.4...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

...

(00:00) [ DONE] Write Model to /content/runs/yolo11n_fall/weights/best_int8.tflite (+00:00)

LiteRT: applying static quantization (int8 weights + int8 activations)...


/usr/local/lib/python3.13/dist-packages/ai_edge_litert/interpreter.py:480: UserWarning: Warning: Enabling `experimental_preserve_all_tensors` with the BUILTIN or AUTO op resolver is intended for debugging purposes only. Be aware that this can significantly increase memory usage by storing all intermediate tensors. If you encounter memory problems or are not actively debugging, consider disabling this option.
  warnings.warn(
Applying Transformations to tensors:: 100%|██████████| 574/574 [00:00<00:00, 21024.26it/s]


Model name: /content/runs/yolo11n_fall/weights/best_int8.tflite
Original model size: 10.06 MiB
Quantized model size: 2.88 MiB
Quantization Ratio: 0.29 (3.5x smaller)
Total time: 209.36 ms
LiteRT: export success ✅ 253.8s, saved as '/content/runs/yolo11n_fall/weights/best_int8.tflite' (2.9 MB)

Export complete (254.3s)
Results saved to /content/runs/yolo11n_fall/weights/best_int8.tflite
Predict:         yolo predict task=detect model=/content/runs/yolo11n_fall/weights/best_int8.tflite imgsz=320 
Validate:        yolo val task=detect model=/content/runs/yolo11n_fall/weights/best_int8.tflite imgsz=320 data=/content/raw_data/data.yaml  
Visualize:       https://netron.app


PosixPath('/content/runs/yolo11n_fall/weights/best_int8.tflite')

In [9]:
!du -sh /content/runs

20M	/content/runs


In [10]:
import shutil
shutil.make_archive('/content/runs_export', 'zip', '/content/runs')

'/content/runs_export.zip'

In [11]:
from google.colab import files
files.download('/content/runs_export.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>